In [ ]:
# @title Imports and Notebook Utilities
# This block is mostly taken from the self-org textures notebook.
import os
import io
import PIL.Image, PIL.ImageDraw
import base64
import zipfile
import json
import requests
import numpy as np
import matplotlib.pylab as pl
import glob
from scipy import ndimage

from IPython.display import Image, HTML, Markdown, clear_output
from tqdm.notebook import tqdm

import warnings
warnings.filterwarnings("ignore")

os.environ['FFMPEG_BINARY'] = 'ffmpeg'
import moviepy.editor as mvp
from moviepy.video.io.ffmpeg_writer import FFMPEG_VideoWriter


def imread(url, max_size=None, mode=None):
    if isinstance(url, str) and url.startswith(('http:', 'https:')):
        # wikimedia requires a user agent
        headers = {
            "User-Agent": "Requests in Colab/0.0 (https://colab.research.google.com/; no-reply@google.com) requests/0.0"
        }
        r = requests.get(url, headers=headers)
        f = io.BytesIO(r.content)
    else:
        f = url
    img = PIL.Image.open(f)
    if max_size is not None:
        img.thumbnail((max_size, max_size), PIL.Image.LANCZOS)
    if mode is not None:
        img = img.convert(mode)
    img = np.float32(img) / 255.0
    return img


def np2pil(a):
    if a.dtype in [np.float32, np.float64]:
        a = np.uint8(np.clip(a, 0, 1) * 255)
    return PIL.Image.fromarray(a)


def imwrite(f, a, fmt=None):
    a = np.asarray(a)
    if isinstance(f, str):
        fmt = f.rsplit('.', 1)[-1].lower()
        if fmt == 'jpg':
            fmt = 'jpeg'
        f = open(f, 'wb')
    np2pil(a).save(f, fmt, quality=95)


def imencode(a, fmt='jpeg'):
    a = np.asarray(a)
    if len(a.shape) == 3 and a.shape[-1] == 4:
        fmt = 'png'
    f = io.BytesIO()
    imwrite(f, a, fmt)
    return f.getvalue()


def im2url(a, fmt='jpeg'):
    encoded = imencode(a, fmt)
    base64_byte_string = base64.b64encode(encoded).decode('ascii')
    return 'data:image/' + fmt.upper() + ';base64,' + base64_byte_string


def imshow(a, fmt='jpeg', id=None):
    return display(Image(data=imencode(a, fmt)), display_id=id)


def grab_plot(close=True):
    """Return the current Matplotlib figure as an image"""
    fig = pl.gcf()
    fig.canvas.draw()
    img = np.array(fig.canvas.renderer._renderer)
    a = np.float32(img[..., 3:] / 255.0)
    img = np.uint8(255 * (1.0 - a) + img[..., :3] * a)  # alpha
    if close:
        pl.close()
    return img



def zoom(img, scale=4):
    img = np.repeat(img, scale, 0)
    img = np.repeat(img, scale, 1)
    return img


class VideoWriter:
    def __init__(self, filename='_autoplay.mp4', fps=30.0, **kw):
        self.writer = None
        self.params = dict(filename=filename, fps=fps, **kw)

    def add(self, img):
        img = np.asarray(img)
        if self.writer is None:
            h, w = img.shape[:2]
            self.writer = FFMPEG_VideoWriter(size=(w, h), **self.params)
        if img.dtype in [np.float32, np.float64]:
            img = np.uint8(img.clip(0, 1) * 255)
        if len(img.shape) == 2:
            img = np.repeat(img[..., None], 3, -1)
        self.writer.write_frame(img)

    def close(self):
        if self.writer:
            self.writer.close()

    def __enter__(self):
        return self

    def __exit__(self, *kw):
        self.close()
        if self.params['filename'] == '_autoplay.mp4':
            self.show()

    def show(self, **kw):
        self.close()
        fn = self.params['filename']
        display(mvp.ipython_display(fn, **kw))

!nvidia-smi -L

In [ ]:
import torch
import torchvision.models as models

torch.set_default_tensor_type('torch.cuda.FloatTensor')

In [ ]:
#@title VGG16 Sliced OT Style Loss
import torch.nn.functional as F

def calc_styles_vgg(imgs, vgg):
    style_layers = [1, 6, 11, 18, 25]
    mean = torch.tensor([0.485, 0.456, 0.406])[:, None, None]
    std = torch.tensor([0.229, 0.224, 0.225])[:, None, None]
    x = (imgs - mean) / std
    b, c, h, w = x.shape
    features = [x.reshape(b, c, h * w)]
    for i, layer in enumerate(vgg[:max(style_layers) + 1]):
        x = layer(x)
        if i in style_layers:
            b, c, h, w = x.shape
            features.append(x.reshape(b, c, h * w))
    return features

def project_sort(x, proj):
    return torch.einsum('bcn,cp->bpn', x, proj).sort()[0]

def ot_loss(source, target, proj_n=32):
    ch, n = source.shape[-2:]
    projs = F.normalize(torch.randn(ch, proj_n), dim=0)
    source_proj = project_sort(source, projs)
    target_proj = project_sort(target, projs)
    target_interp = F.interpolate(target_proj, n, mode='nearest')
    return (source_proj - target_interp).square().sum()

def create_rotation_invariant_loss(vgg, target_img, n_rotations=64):
    """Create rotation-invariant loss by comparing against rotated versions of target."""
    # Generate rotated versions of target
    target_styles = []
    target_np = target_img[0].permute(1, 2, 0).cpu().numpy()
    
    for angle in np.linspace(0.0, 360, n_rotations + 1)[:-1]:
        rotated = ndimage.rotate(target_np, angle, reshape=False, mode='wrap', order=1)
        rotated_torch = torch.tensor(rotated).permute(2, 0, 1).unsqueeze(0)
        with torch.no_grad():
            features = calc_styles_vgg(rotated_torch, vgg)
            target_styles.append(features)
    
    def loss_f(imgs):
        xx = calc_styles_vgg(imgs, vgg)
        # Compute loss against all rotations and take minimum
        min_loss = float('inf')
        for target_features in target_styles:
            loss = sum(ot_loss(x, y) for x, y in zip(xx, target_features))
            min_loss = torch.minimum(min_loss, loss)
        return min_loss
    
    return loss_f

In [ ]:
#@title Load VGG and Target image {vertical-output: true}
vgg = models.vgg16(weights='IMAGENET1K_V1').features

from google.colab import files

print("Please upload your style image:")
uploaded = files.upload()

filename = list(uploaded.keys())[0]
print(f'Using uploaded file: "{filename}"')

style_img = imread(io.BytesIO(uploaded[filename]), max_size=128)
style_img_torch = torch.tensor(style_img).permute(2, 0, 1).unsqueeze(0)

print("Creating rotation-invariant loss function (this may take a moment)...")
with torch.no_grad():
    loss_fn = create_rotation_invariant_loss(vgg, style_img_torch, n_rotations=64)
print("Loss function ready!")
imshow(style_img)

In [ ]:
#@title Reaction-Diffusion CA Architecture

def pad_circular(x, pad=1):
    """Circular padding for 2D spatial dimensions."""
    # Pad height (dim 2)
    x = torch.cat([x[:, :, -pad:], x, x[:, :, :pad]], dim=2)
    # Pad width (dim 3)
    x = torch.cat([x[:, :, :, -pad:], x, x[:, :, :, :pad]], dim=3)
    return x

def laplacian(x):
    """Apply Laplacian filter to all channels independently."""
    # Laplacian kernel (unnormalized, as in RD notebook)
    lap = torch.tensor([[1.0, 2.0, 1.0], 
                        [2.0, -12.0, 2.0], 
                        [1.0, 2.0, 1.0]]) / 16.0
    
    b, ch, h, w = x.shape
    # Reshape for depthwise conv
    kernel = lap.view(1, 1, 3, 3).repeat(ch, 1, 1, 1)
    
    # Apply circular padding
    x_padded = pad_circular(x, pad=1)
    
    # Depthwise convolution
    y = F.conv2d(x_padded, kernel, groups=ch)
    return y


class ReactionDiffusionCA(torch.nn.Module):
    def __init__(self, chn=32, hidden_n=128, noise_level=0.1):
        super().__init__()
        self.chn = chn
        self.register_buffer("noise_level", torch.tensor([noise_level]))
        
        # Reaction network (learns nonlinear dynamics)
        self.w1 = torch.nn.Conv2d(chn, hidden_n, 1, bias=True)
        self.w2 = torch.nn.Conv2d(hidden_n, chn, 1, bias=False)
        
        # Initialize reaction network to zero (start with pure diffusion)
        torch.nn.init.xavier_normal_(self.w1.weight, gain=0.2)
        torch.nn.init.zeros_(self.w2.weight)
        
        # Diffusion coefficients (multi-scale, as in TF notebook)
        # 4 groups with different diffusion rates
        diff_coef = torch.tensor([0.125, 0.25, 0.5, 1.0]).repeat(chn // 4)
        if chn % 4 != 0:
            # Pad if not divisible by 4
            diff_coef = torch.cat([diff_coef, torch.ones(chn % 4)])
        self.register_buffer("diff_coef", diff_coef)

    def forward(self, x, r=1.0, d=1.0, noise=None):
        """
        Forward pass with explicit reaction-diffusion dynamics.
        
        Args:
            x: state tensor [batch, channels, height, width]
            r: reaction rate multiplier (default 1.0)
            d: diffusion rate multiplier (default 1.0)
            noise: optional noise level for training
        """
        if noise is not None:
            x = x + torch.randn_like(x) * noise
        
        # Diffusion term: Laplacian with per-channel coefficients
        diff = laplacian(x) * self.diff_coef[None, :, None, None]
        
        # Reaction term: Learned nonlinear dynamics with Swish activation
        y = self.w1(x)
        y = y * torch.sigmoid(y * 5.0)  # Swish activation
        react = self.w2(y)
        
        # Explicit reaction-diffusion update
        x = x + diff * d + react * r
        
        return x

    def seed(self, n, h=128, w=128, spot_prob=0.005, spread=3.0):
        """
        Create seed states with scattered gaussian blobs (as in TF notebook).
        Only RGB channels (0-2) are initialized; hidden channels start at 0.
        """
        # Random spots
        x = (torch.rand(n, h, w, 1) < spot_prob).float()
        
        # Gaussian blur (approximate with multiple avg pooling)
        x_np = x.cpu().numpy()
        from scipy.ndimage import gaussian_filter
        x_np = gaussian_filter(x_np, [0.0, spread, spread, 0.0], mode='wrap')
        x = torch.from_numpy(x_np).float()
        
        # Scale by spread^2 (as in TF notebook)
        x = x * (spread ** 2)
        
        # Repeat for RGB channels
        x = x.repeat(1, 1, 1, 3)
        
        # Pad with zeros for hidden channels
        if self.chn > 3:
            padding = torch.zeros(n, h, w, self.chn - 3)
            x = torch.cat([x, padding], dim=-1)
        
        # Convert to NCHW format
        x = x.permute(0, 3, 1, 2)
        
        return x


def to_rgb(s):
    """Extract RGB channels and shift to [0, 1] range."""
    return s[..., :3, :, :] + 0.5


# Test model creation
model = ReactionDiffusionCA(chn=32, hidden_n=128)
param_n = sum(p.numel() for p in model.parameters())
print('ReactionDiffusionCA param count:', param_n)
print('Diffusion coefficients:', model.diff_coef.cpu().numpy())

In [ ]:
#@title Setup Training
from google.colab import files

print("Please upload your checkpoint file (.pt), or click 'Cancel' to initialize from scratch:")
uploaded = files.upload()

if uploaded:
    filename = list(uploaded.keys())[0]
    print(f'Loading checkpoint from: "{filename}"')
    checkpoint = torch.load(io.BytesIO(uploaded[filename]))
    
    # Restore model
    model = ReactionDiffusionCA(chn=32, hidden_n=128)
    model.load_state_dict(checkpoint['model_state_dict'])
    
    # Restore optimizer
    opt = torch.optim.Adam(model.parameters(), 1e-3, capturable=True)
    opt.load_state_dict(checkpoint['optimizer_state_dict'])
    
    # Restore learning rate scheduler
    lr_sched = torch.optim.lr_scheduler.MultiStepLR(opt, [1000, 2000], 0.3)
    lr_sched.load_state_dict(checkpoint['scheduler_state_dict'])
    
    # Restore training state
    loss_log = checkpoint['loss_log']
    pool = checkpoint['pool']
    start_iteration = checkpoint['iteration'] + 1
    
    print(f'Resuming from iteration {start_iteration}')
    print(f'Loss history length: {len(loss_log)}')
    print(f'Current learning rate: {lr_sched.get_last_lr()[0]:.2e}')
else:
    print('Initializing new ReactionDiffusionCA model...')
    model = ReactionDiffusionCA(chn=32, hidden_n=128)
    opt = torch.optim.Adam(model.parameters(), 1e-3, capturable=True)
    lr_sched = torch.optim.lr_scheduler.MultiStepLR(opt, [1000, 2000], 0.3)
    loss_log = []
    with torch.no_grad():
        pool = model.seed(256)
    start_iteration = 0
    
print('Pool shape:', pool.shape)

In [ ]:
# @title Training loop {vertical-output: true}

# Training parameters
total_iterations = 5000
checkpoint_interval = 100  # Save checkpoint every N iterations
auto_checkpoint_file = 'rd_checkpoint_auto.pt'

try:
    for i in range(start_iteration, total_iterations):
        with torch.no_grad():
            batch_idx = np.random.choice(len(pool), 4, replace=False)
            s = pool[batch_idx]
            if i % 8 == 0:
                s[:1] = model.seed(1)
        
        step_n = np.random.randint(32, 96)
        for k in range(step_n):
            s = model(s, r=1.0, d=1.0, noise=0.05)  # reaction and diffusion rates

        overflow_loss = (s - s.clamp(-1.0, 1.0)).abs().sum()
        loss = loss_fn(to_rgb(s)) + overflow_loss
        
        with torch.no_grad():
            loss.backward()
            for p in model.parameters():
                p.grad /= (p.grad.norm() + 1e-8)  # normalize gradients
            opt.step()
            opt.zero_grad()
            lr_sched.step()
            pool[batch_idx] = s  # update pool

            loss_log.append(loss.item())
            if i % 5 == 0:
                display(Markdown(f'''
        step_n: {len(loss_log)}
        iteration: {i}/{total_iterations}
        loss: {loss.item():.4f}
        lr: {lr_sched.get_last_lr()[0]:.2e}'''), display_id='stats')
            if i % 32 == 0:
                pl.plot(loss_log, '.', alpha=0.1)
                pl.yscale('log')
                pl.ylim(np.min(loss_log), loss_log[0])
                pl.tight_layout()
                imshow(grab_plot(), id='log')
                imgs = to_rgb(s).permute([0, 2, 3, 1]).cpu()
                imshow(np.hstack(imgs), id='batch')
            
            # Auto-save checkpoint periodically
            if i % checkpoint_interval == 0 and i > 0:
                checkpoint = {
                    'iteration': i,
                    'model_state_dict': model.state_dict(),
                    'optimizer_state_dict': opt.state_dict(),
                    'scheduler_state_dict': lr_sched.state_dict(),
                    'loss_log': loss_log,
                    'pool': pool,
                }
                torch.save(checkpoint, auto_checkpoint_file)
                print(f'Auto-checkpoint saved at iteration {i}')

    print('Training completed!')
    
except KeyboardInterrupt:
    print('\\n\\nTraining interrupted by user!')
    print(f'Stopped at iteration {i}/{total_iterations}')
    
    # Save checkpoint on interruption
    checkpoint = {
        'iteration': i,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': opt.state_dict(),
        'scheduler_state_dict': lr_sched.state_dict(),
        'loss_log': loss_log,
        'pool': pool,
    }
    interrupt_file = f'rd_checkpoint_interrupted_iter{i}.pt'
    torch.save(checkpoint, interrupt_file)
    print(f'\\nCheckpoint saved to: {interrupt_file}')
    print('You can resume training by uploading this file in cell 5.')
    
    # Offer to download
    from google.colab import files
    files.download(interrupt_file)

In [ ]:
#@title Save Checkpoint (Full Training State)
import torch
from google.colab import files

# Save complete checkpoint with optimizer state and training history
checkpoint_filename = 'rd_checkpoint.pt'

checkpoint = {
    'iteration': len(loss_log) - 1 if loss_log else 0,
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': opt.state_dict(),
    'scheduler_state_dict': lr_sched.state_dict(),
    'loss_log': loss_log,
    'pool': pool,
}

torch.save(checkpoint, checkpoint_filename)
print(f"Full checkpoint saved as '{checkpoint_filename}'")
print(f"Iteration: {checkpoint['iteration']}")
print(f"Loss log entries: {len(loss_log)}")
print(f"Current LR: {lr_sched.get_last_lr()[0]:.2e}")
print("\\nYou can resume training by uploading this file in cell 5.")
files.download(checkpoint_filename)

# Also save just the model weights (smaller file, for inference only)
weights_filename = 'rd_weights.pt'
torch.save(model.state_dict(), weights_filename)
print(f"\\nModel weights (inference only) saved as '{weights_filename}'")
files.download(weights_filename)